# Unitree G1 — PPO + AIRL từ GitHub
Source được clone từ `nvhiep249/mjlab`, branch `feature/kaggle-airl-16k`.
Gắn Dataset `nvhiep2409/airl-ppo`, chọn GPU và bật Internet. Không cần upload source ZIP.
Expert/checkpoint đọc từ `/kaggle/input/datasets/nvhiep2409/airl-ppo`.
Notebook ghi commit Git thực tế và kiểm tra source/data SHA256 trước train.
Quét 16.384–32.768 env, 4/8 minibatches; giữ24steps và5epochs, tối thiểu15% VRAM trống.
Hai arms PPO/AIRL dùng cùng selection, seed, common checkpoint và transition budget.


In [ ]:
import ctypes.util
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/nvhiep249/mjlab.git"
REPO_BRANCH = "feature/kaggle-airl-16k"
REPO_COMMIT = None  # Set a full commit SHA to replay a saved run exactly.
INPUT_ROOT = Path("/kaggle/input/datasets/nvhiep2409/airl-ppo")
OUTPUT = Path("/kaggle/working/outputs")
GPU = 0
SESSION = "pilot01"  # Change for a new run/chunk, preserving previous outputs.
SOURCE = Path("/tmp") / f"mjlab-github-{SESSION}"
TRANSITIONS = 23592960
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"
os.environ["PYTHONUTF8"] = "1"
os.environ["MUJOCO_GL"] = "egl"
OUTPUT.mkdir(parents=True, exist_ok=True)


def find_input(names):
  for name in names:
    matches = list(INPUT_ROOT.rglob(name))
    if len(matches) > 1:
      raise RuntimeError(f"Ambiguous input {name}: {matches}; set the path explicitly")
    if matches:
      return matches[0]
  raise FileNotFoundError(
    f"Attach nvhiep2409/airl-ppo; missing {names} under {INPUT_ROOT}"
  )


EXPERT = find_input(("airl_expert_1p2_deterministic_v1.pt",))
COMMON = find_input(("ppo_common_999.pt", "model_999.pt"))
INPUT = EXPERT.parent
print("Expert:", EXPERT, "Common PPO:", COMMON)
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
def digest(path):
  hasher = hashlib.sha256()
  with path.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
      hasher.update(chunk)
  return hasher.hexdigest()


DATA_FILES = {
  "expert": {
    "sha256": "3cf8b910c4957fc80d257762d55945e5777411ba2785659dfb29dd08d69e72e8",
    "bytes": 382726465,
  },
  "common_ppo": {
    "sha256": "b318a6da5db37a6795e4a943d0d52eaa75743bd403341ecf47b10b3f20972dc8",
    "bytes": 5316259,
  },
}
for name, path in (("expert", EXPERT), ("common_ppo", COMMON)):
  expected = DATA_FILES[name]
  if path.stat().st_size != expected["bytes"] or digest(path) != expected["sha256"]:
    raise RuntimeError(
      f"Wrong qualified input: {name}, {path}; do not train on a different file"
    )
(OUTPUT / f"{SESSION}_data_manifest.json").write_text(json.dumps(DATA_FILES, indent=2))
print("Expert and common PPO SHA256 verified")

In [ ]:
if SOURCE.exists():
  raise FileExistsError(
    f"Use a new SESSION or a fresh runtime; source already exists: {SOURCE}"
  )
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
subprocess.run(
  [
    "git",
    "clone",
    "--config",
    "core.autocrlf=false",
    "--depth",
    "1",
    "--single-branch",
    "--branch",
    REPO_BRANCH,
    REPO_URL,
    str(SOURCE),
  ],
  check=True,
  env=git_env,
)
if REPO_COMMIT is not None:
  subprocess.run(
    ["git", "fetch", "--depth", "1", "origin", REPO_COMMIT],
    cwd=SOURCE,
    check=True,
    env=git_env,
  )
  subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=SOURCE, check=True)
source_commit = subprocess.check_output(
  ["git", "rev-parse", "HEAD"], cwd=SOURCE, text=True
).strip()
if REPO_COMMIT is not None and source_commit != REPO_COMMIT:
  raise RuntimeError("Requested source commit was not checked out")
manifest = json.loads(
  (SOURCE / "scripts/cloud/kaggle_source_manifest.json").read_text()
)
for relative, expected in manifest["source_files"].items():
  target = (SOURCE / relative).resolve()
  if not target.is_relative_to(SOURCE.resolve()) or digest(target) != expected:
    raise RuntimeError(f"Source hash mismatch: {relative}")
provenance = {
  "repo": REPO_URL,
  "branch": REPO_BRANCH,
  "commit": source_commit,
  "source_files": manifest["source_files"],
  "data_files": DATA_FILES,
}
(OUTPUT / f"{SESSION}_github_source.json").write_text(json.dumps(provenance, indent=2))
print("Verified GitHub source:", source_commit)

In [ ]:
if not shutil.which("uv"):
  subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
if not ctypes.util.find_library("EGL"):
  subprocess.run(["apt-get", "update", "-qq"], check=True)
  subprocess.run(["apt-get", "install", "-y", "libegl1"], check=True)
subprocess.run(
  ["uv", "sync", "--locked", "--no-dev", "--extra", "cu128", "--python", "3.11"],
  cwd=SOURCE,
  check=True,
)
subprocess.run(
  [
    "uv",
    "run",
    "--no-sync",
    "python",
    "-c",
    "import importlib.metadata as m; print({p:m.version(p) for p in "
    "['torch','warp-lang','mujoco','mujoco-warp','rsl-rl-lib']})",
  ],
  cwd=SOURCE,
  check=True,
)

TOOL = "scripts/cloud/kaggle_airl.py"


def cloud(*arguments):
  return subprocess.run(
    ["uv", "run", "--no-sync", "python", TOOL, *map(str, arguments)],
    cwd=SOURCE,
    check=True,
  )

## CUDA / train / resume smoke
5 updates/arm ở 1.024 env là integration check. AIRL cập nhật actor/critic và g/h;
không phải bằng chứng locomotion tốt. Smoke cũng chạy Torch CUDA op và Warp G1 thật.
Nếu OOM, nonfinite, kernel/contact overflow hoặc density contract lỗi: dừng và đọc log.


In [ ]:
for arm in ("ppo", "airl"):
  target = OUTPUT / f"{SESSION}_smoke_{arm}"
  cloud(
    "worker",
    "--input",
    INPUT,
    "--checkpoint",
    COMMON,
    "--output",
    target,
    "--result",
    target / "run_summary.json",
    "--envs",
    1024,
    "--minibatches",
    4,
    "--updates",
    5,
    "--warmup",
    0,
    "--gpu",
    GPU,
    "--arm",
    arm,
  )

smoke = json.loads((OUTPUT / f"{SESSION}_smoke_airl/run_summary.json").read_text())
checkpoint = max(
  Path(smoke["run_dir"]).glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[-1])
)
target = OUTPUT / f"{SESSION}_resume_smoke"
cloud(
  "worker",
  "--input",
  INPUT,
  "--checkpoint",
  checkpoint,
  "--resume",
  "--output",
  target,
  "--result",
  target / "run_summary.json",
  "--envs",
  1024,
  "--minibatches",
  4,
  "--updates",
  1,
  "--warmup",
  0,
  "--gpu",
  GPU,
  "--arm",
  "airl",
)

## Evaluate common PPO baseline trước khi train
Đây là checkpoint PPO999 khởi tạo learner; expert dataset do teacher35550 tạo đã qualify riêng.
Baseline FAIL nghĩa là PPO999 chưa đạt task gate, không phải expert dataset bị loại hay lỗi chương trình.
Giữ report gốc và cùng COMMON cho cả hai arms; không hạ gate hoặc thay teacher vào một arm.
Nếu metrics hợp lệ, tiếp tục sweep rồi paired pilot để đo mức cải thiện từ baseline chưa đạt.
Nếu base đã pass thì time-to-gate tại đây bằng0; sau pilot so stability/RMSE/falls.
Dataset chỉ được qualify cho1.2m/s thẳng; nguyên nhân FAIL nằm trong gate.failures và bins.


In [ ]:
def evaluate(checkpoint, output, seed, role="endpoint"):
  subprocess.run(
    [
      "uv",
      "run",
      "--no-sync",
      "gail-evaluate-expert",
      "--task",
      manifest["task"],
      "--checkpoint-file",
      str(checkpoint),
      "--output-file",
      str(output),
      "--evaluation-role",
      role,
      "--device",
      "cuda:0",
      "--episodes-per-command",
      "100",
      "--seed",
      str(seed),
      "--lin-vel-x",
      "1.2",
      "--lin-vel-y",
      "0.0",
      "--ang-vel-z",
      "0.0",
    ],
    cwd=SOURCE,
    check=True,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": str(GPU)},
  )
  return json.loads(output.read_text())


baseline = evaluate(
  COMMON, OUTPUT / f"{SESSION}_common_base_seed42005.json", 42005, role="baseline"
)
print("Common PPO baseline gate:", baseline["gate"])
if not baseline["gate"]["passed"]:
  print(
    "Baseline chưa đạt task gate. Expert dataset được qualify riêng; "
    "giữ report này và tiếp tục sweep + paired pilot nếu metrics hợp lệ."
  )

## Benchmark toàn vòng AIRL + PPO
Mỗi cấu hình process riêng: 3 warmup + 8 updates đo, bỏ checkpoint/ONNX khi benchmark.
Đo median update wall time có CUDA synchronization, gồm rollout + D + PPO.
`nvidia-smi` sampling đo tổng VRAM gồm Warp; bổ sung Torch peak-reserved.
Đây là peak được lấy mẫu, vẫn phải theo dõi run dài. Chỉ chọn >16.000 env với 15% headroom.
Timeout/OOM của một candidate được giữ log và chuyển candidate tiếp theo.


In [ ]:
SWEEP = OUTPUT / f"{SESSION}_sweep"
cloud(
  "sweep",
  "--input",
  INPUT,
  "--checkpoint",
  COMMON,
  "--output",
  SWEEP,
  "--arm",
  "airl",
  "--gpu",
  GPU,
  "--envs",
  16384,
  20480,
  24576,
  32768,
  "--minibatches",
  4,
  8,
  "--warmup",
  3,
  "--measure",
  8,
)
SELECTED = SWEEP / "selected.json"
selected = json.loads(SELECTED.read_text())
print(json.dumps(selected, indent=2))
print("Pilot updates/arm:", TRANSITIONS // selected["rollout_transitions"])

## Pilot cùng budget
Run lần đầu dùng COMMON và không `--resume`. Hai arm cùng env/minibatches/seed/budget.
Continuation đo lợi ích bổ sung AIRL từ policy đã có; không chứng minh train fresh nhanh hơn.
Một D update/batch1024 vẫn giữ nguyên; log D/density/reward để kiểm tra mất cân bằng.
Nếu một chunk dự kiến vượt 2 giờ, giảm TRANSITIONS về cùng bội rollout cho cả hai arm.
Để chạy tiếp session sau: gắn outputs cũ làm input, đặt checkpoint riêng từng arm,
dùng `--resume`, giữ selected/config/seed và cùng lịch chunk. AIRL checkpoint không dùng cho control.
Simulator reset khi resume; không có exact simulator-state restore.
Chạy lại cell: reuse arm hoàn tất nếu config/budget khớp; arm dở chạy lại từ COMMON trong attempt mới, giữ output cũ.


In [ ]:
estimated_seconds = selected["median_update_seconds"] * (
  TRANSITIONS // selected["rollout_transitions"]
)
if estimated_seconds > 2 * 3600:
  raise RuntimeError("Reduce TRANSITIONS for both arms before starting this chunk")
for arm in ("ppo", "airl"):
  target = OUTPUT / f"{SESSION}_pilot_{arm}"
  cloud(
    "train",
    "--input",
    INPUT,
    "--checkpoint",
    COMMON,
    "--selected",
    SELECTED,
    "--output",
    target,
    "--transitions",
    TRANSITIONS,
    "--arm",
    arm,
    "--gpu",
    GPU,
    "--reuse-completed",
    "--restart-incomplete",
  )

In [ ]:
for arm in ("ppo", "airl"):
  summary = json.loads((OUTPUT / f"{SESSION}_pilot_{arm}/run_summary.json").read_text())
  checkpoint = max(
    Path(summary["run_dir"]).glob("model_*.pt"),
    key=lambda p: int(p.stem.split("_")[-1]),
  )
  for seed in (42005, 42006):
    evaluate(checkpoint, OUTPUT / f"{SESSION}_{arm}_eval_seed{seed}.json", seed)

hashes = {
  path.relative_to(OUTPUT).as_posix(): digest(path)
  for path in OUTPUT.rglob("*")
  if path.is_file() and path.name != "output_hashes.json"
}
(OUTPUT / "output_hashes.json").write_text(json.dumps(hashes, indent=2))
print("Save notebook version with outputs, then download /kaggle/working/outputs")